# Billion-Dollar Business Idea Framework
## Market Research Automation | TAM/SAM/SOM Calculator | Competitor Analysis

Systematic framework for validating business ideas using data.

---

In [ ]:
# CELL 1: Setup
import subprocess, sys
for p in ['numpy','pandas','matplotlib']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict
OUTPUT_DIR = Path('outputs'); OUTPUT_DIR.mkdir(exist_ok=True)
print('Framework Ready!')

In [ ]:
# CELL 2: Business Idea Validator
@dataclass
class BusinessIdea:
    name: str
    description: str
    target_market: str
    revenue_model: str
    differentiator: str
    
    # Market sizing (in millions USD)
    tam: float = 0  # Total Addressable Market
    sam: float = 0  # Serviceable Addressable Market
    som: float = 0  # Serviceable Obtainable Market
    
    competitors: List[Dict] = field(default_factory=list)
    score: float = 0.0

class IdeaValidator:
    """Score and validate business ideas systematically."""
    
    CRITERIA = {
        'market_size': {'weight': 0.20, 'desc': 'TAM > $10B = 10, > $1B = 7, > $100M = 5'},
        'competition': {'weight': 0.15, 'desc': 'Blue ocean = 10, Few competitors = 7, Crowded = 3'},
        'scalability': {'weight': 0.20, 'desc': 'Software/digital = 10, Services = 5, Physical = 3'},
        'moat': {'weight': 0.15, 'desc': 'Network effects = 10, IP/data = 7, Brand = 5, None = 2'},
        'timing': {'weight': 0.10, 'desc': 'Perfect timing = 10, Good = 7, Early = 5, Late = 3'},
        'team_fit': {'weight': 0.10, 'desc': 'Deep expertise = 10, Some experience = 6, Learning = 3'},
        'revenue_clarity': {'weight': 0.10, 'desc': 'Proven model = 10, Similar exists = 7, Novel = 4'},
    }
    
    def score_idea(self, idea: BusinessIdea, scores: Dict[str, int]) -> float:
        total = sum(scores.get(c, 5) * self.CRITERIA[c]['weight'] for c in self.CRITERIA)
        idea.score = round(total, 2)
        return idea.score
    
    def calculate_tam_sam_som(self, idea, total_market_users, avg_revenue_per_user,
                               serviceable_pct=0.3, obtainable_pct=0.05):
        idea.tam = total_market_users * avg_revenue_per_user / 1e6
        idea.sam = idea.tam * serviceable_pct
        idea.som = idea.sam * obtainable_pct
        return idea.tam, idea.sam, idea.som
    
    def visualize(self, ideas: List[BusinessIdea]):
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        # Score comparison
        names = [i.name for i in ideas]
        scores = [i.score for i in ideas]
        colors = ['green' if s >= 7 else 'orange' if s >= 5 else 'red' for s in scores]
        axes[0].barh(names, scores, color=colors)
        axes[0].set_title('Idea Scores (10 = Perfect)')
        axes[0].set_xlim(0, 10); axes[0].axvline(7, color='green', linestyle='--', alpha=0.3, label='Good')
        
        # Market sizing
        x = np.arange(len(names)); w = 0.25
        axes[1].bar(x - w, [i.tam for i in ideas], w, label='TAM', color='#2196F3')
        axes[1].bar(x, [i.sam for i in ideas], w, label='SAM', color='#4CAF50')
        axes[1].bar(x + w, [i.som for i in ideas], w, label='SOM', color='#FF9800')
        axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=15)
        axes[1].set_title('Market Size ($M)'); axes[1].legend()
        
        plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'idea_analysis.png', dpi=150); plt.show()

# Demo with sample ideas
validator = IdeaValidator()

ideas = [
    BusinessIdea("AI Tutor", "Personalized AI tutoring for K-12", "Students & parents", "Subscription", "Adaptive learning"),
    BusinessIdea("CarbonTrack", "Carbon footprint tracker for businesses", "SMBs", "SaaS + compliance", "Automated reporting"),
    BusinessIdea("SkillSwap", "P2P skill exchange marketplace", "Young professionals", "Freemium + commission", "AI matching"),
]

# Score each
for idea, scores in zip(ideas, [
    {'market_size': 8, 'competition': 5, 'scalability': 9, 'moat': 7, 'timing': 8, 'team_fit': 6, 'revenue_clarity': 8},
    {'market_size': 9, 'competition': 6, 'scalability': 8, 'moat': 8, 'timing': 9, 'team_fit': 5, 'revenue_clarity': 7},
    {'market_size': 7, 'competition': 7, 'scalability': 8, 'moat': 6, 'timing': 7, 'team_fit': 7, 'revenue_clarity': 6},
]):
    validator.calculate_tam_sam_som(idea, 200e6, 50, 0.1, 0.02)
    validator.score_idea(idea, scores)
    print(f"{idea.name}: Score={idea.score} | TAM=${idea.tam:.0f}M SAM=${idea.sam:.0f}M SOM=${idea.som:.0f}M")

validator.visualize(ideas)
print("\nP6 Business Idea Framework - COMPLETE!")